<a href="https://colab.research.google.com/github/AvanindraBose/CampusX-courses/blob/main/OHE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Apply OHE**

In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('cars.csv')

In [4]:
df.head(5)

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [7]:
df['brand'].value_counts()

,count
brand,
Maruti,2448
Hyundai,1415
Mahindra,772
Tata,734
Toyota,488
Honda,467
Ford,397
Chevrolet,230
Renault,228


In [8]:
df['fuel'].value_counts()

,count
fuel,
Diesel,4402
Petrol,3631
CNG,57
LPG,38


In [11]:
df['owner'].unique()

array(['First Owner', 'Second Owner', 'Third Owner',
       'Fourth & Above Owner', 'Test Drive Car'], dtype=object)

**PANDAS OHE**

In [19]:
pd.get_dummies(df,columns=['fuel','owner'],dtype=np.int32,drop_first=True)

,brand,km_driven,selling_price,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,1,0,0,0,0,0,0
1,Skoda,120000,370000,1,0,0,0,1,0,0
2,Honda,140000,158000,0,0,1,0,0,0,1
3,Hyundai,127000,225000,1,0,0,0,0,0,0
4,Maruti,120000,130000,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,0,0,1,0,0,0,0
8124,Hyundai,119000,135000,1,0,0,1,0,0,0
8125,Maruti,120000,382000,1,0,0,0,0,0,0
8126,Tata,25000,290000,1,0,0,0,0,0,0


Problem with this Approach is that Its not inpleace meaning the changes that are made is not affecting df , unless we assign df = pd.get_dummies()
Hence We use Sk learn OHE

In [20]:
from sklearn.preprocessing import OneHotEncoder

In [25]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.iloc[:,:4] , df.iloc[:,-1],test_size=0.2,random_state=42)

In [43]:
ohe = OneHotEncoder(drop = 'first' ,dtype=np.int32)

In [44]:
X_train_new = ohe.fit_transform(X_train[['fuel','owner']]).toarray()

In [45]:
X_test_new = ohe.transform(X_test[['fuel','owner']]).toarray()

In [47]:
X_train_new.shape

(6502, 7)

In [46]:
np.hstack((X_train[['brand','km_driven']].values,X_train_new))

array([['Tata', 2560, 0, ..., 0, 0, 0],
       ['Honda', 80000, 0, ..., 1, 0, 0],
       ['Hyundai', 150000, 1, ..., 0, 0, 0],
       ...,
       ['Hyundai', 35000, 0, ..., 0, 0, 0],
       ['Maruti', 27000, 1, ..., 0, 0, 0],
       ['Maruti', 70000, 0, ..., 1, 0, 0]], dtype=object)

**OHE Using Column Trnasformer**

In [48]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [49]:
X_train,X_test,y_train,y_test = train_test_split(df.iloc[:,:4] , df.iloc[:,-1],test_size=0.2,random_state=42)

In [66]:
preprocess = ColumnTransformer(
    transformers=[
        ('ohe' , OneHotEncoder(dtype = np.int32 , drop='first'),['fuel' , 'owner'])
    ],
    remainder='passthrough'
)

In [68]:
preprocess.fit_transform(X_train).shape

(6502, 9)

In [70]:
preprocess.transform(X_test)

array([[0, 0, 1, ..., 1, 'Honda', 110000],
       [1, 0, 0, ..., 0, 'Tata', 291977],
       [1, 0, 0, ..., 0, 'Maruti', 70000],
       ...,
       [0, 0, 1, ..., 0, 'Hyundai', 54043],
       [0, 0, 1, ..., 0, 'Tata', 70000],
       [0, 0, 1, ..., 0, 'Chevrolet', 110000]], dtype=object)

In [57]:
clf = Pipeline(
    steps = [
        ('pre',preprocess),
    ]
)

In a typical machine learning pipeline, all data transformations such as encoding (label, one-hot, ordinal), imputation (mean, median, mode), and scaling should always be performed after splitting the data into training and testing sets.

The main reason is to prevent data leakage — a situation where information from the test set inadvertently influences the training process. For example, if I compute the mean of a feature to impute missing values before the split, that mean will be influenced by both train and test data, which violates the independence of the test set.

In real-world scenarios, the model won't have access to future or unseen data during training, so it's important to simulate that by ensuring that transformations are fit only on the training data and applied to the test data.

Here's what I usually do:

First, I split the data into X_train and X_test.

Then, for transformations like SimpleImputer, OneHotEncoder, LabelEncoder, or even StandardScaler, I call .fit() on X_train, and .transform() on both X_train and X_test.

This ensures the model is trained only using patterns present in the training data and evaluated fairly on truly unseen data.

Additionally, for production-ready pipelines, I use ColumnTransformer and Pipeline from sklearn, which automate this flow and ensure the transformations are applied consistently during model deployment.

**Imp Interview question-How Removing One Column reduces Dummy Trap Variable**

When we perform one-hot encoding on a categorical variable with k unique categories, it creates k new binary columns. However, these columns are not independent — one of them can always be predicted from the others. This creates a problem called multicollinearity.

To avoid this, we typically drop one of the dummy variables. This technique is known as avoiding the dummy variable trap. It ensures that the model can still learn the necessary patterns without introducing redundant information.

For example, if we have a 'Color' column with three categories: Red, Green, Blue — one-hot encoding gives us three columns:

Color_Red

Color_Green

Color_Blue

But if we know any two of them, the third is always implied:

ini
Copy
Edit
Color_Blue = 1 - (Color_Red + Color_Green)
This linear dependency causes issues in models like linear regression and logistic regression, which assume that input features are not perfectly collinear.

So by dropping one column — say, 'Color_Blue' — we retain all the information but eliminate multicollinearity.

Models like tree-based models (e.g., XGBoost, Random Forest) aren't as sensitive to multicollinearity, but it's still a good practice for interpretability and model stability.

In practice, we use:

python
Copy
Edit
OneHotEncoder(drop='first')
or

python
Copy
Edit
pd.get_dummies(df, drop_first=True)
to automatically drop one dummy column and avoid this trap.